<a href="https://colab.research.google.com/github/mahdisahad/WOA-Feature-Selection/blob/main/notebooks/01_Data_Exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="text-align: right">

# نوت‌بوک ۱: بررسی و شناخت داده (Data Exploration)

هدف این نوت‌بوک این است که قبل از هر مدل‌سازی، دیتاست فعال (`ACTIVE_DATASET` در `dataset_config.py`) را بشناسیم: ابعاد، کلاس‌ها، داده‌ی گم‌شده، ویژگی‌های ثابت و تکراری و مقیاس مقادیر. در پایان داده به‌صورت CSV ذخیره می‌شود تا نوت‌بوک‌های بعدی سریع‌تر آن را بخوانند.

</div>

<div dir="rtl" style="text-align: right">

### نصب کتابخانه‌ها
کتابخانه‌های مورد نیاز (numpy، pandas، scikit-learn، matplotlib، scipy) نصب می‌شوند. در Colab معمولاً از قبل نصب‌اند و پیام «already satisfied» می‌دهد.

</div>

In [ ]:
!pip install numpy pandas scikit-learn matplotlib scipy

<div dir="rtl" style="text-align: right">

### اتصال به Google Drive
Drive به محیط Colab وصل می‌شود تا به پوشه‌ی پروژه (`WOA_Project`) شامل دیتاست‌ها و فایل `dataset_config.py` دسترسی داشته باشیم. `force_remount=True` اتصال را از نو برقرار می‌کند.

</div>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

<div dir="rtl" style="text-align: right">

### تعریف مسیرها و وارد کردن تنظیمات دیتاست
- `PROJECT_PATH` مسیر اصلی پروژه در Drive است.
- با `from dataset_config import *` توابع و تنظیمات وابسته به دیتاست (مثل `load_dataset` و `ACTIVE_DATASET`) وارد می‌شوند. برای عوض کردن دیتاست فقط همان فایل را تغییر می‌دهیم.
- در پایان وجود پوشه‌های پروژه، دیتاست‌ها، نوت‌بوک‌ها و نتایج بررسی و چاپ می‌شود.

</div>

In [ ]:
from pathlib import Path

PROJECT_PATH = Path(
    "/content/drive/MyDrive/WOA_Project"
)

import sys
sys.path.append(str(PROJECT_PATH))
from dataset_config import *   # <-- dataset-specific code lives only here
print("Active dataset:", ACTIVE_DATASET)

NOTEBOOK_PATH = PROJECT_PATH / "notebooks"

RESULTS_PATH = PROJECT_PATH / "results"

print("Project exists:", PROJECT_PATH.exists())
print("Datasets folder exists:", (PROJECT_PATH / "datasets").exists())
print("Notebooks exists:", NOTEBOOK_PATH.exists())
print("Results exists:", RESULTS_PATH.exists())

<div dir="rtl" style="text-align: right">

### وارد کردن کتابخانه‌ها و تنظیمات اولیه
numpy و pandas برای محاسبات و جدول داده، و matplotlib و seaborn برای رسم نمودار. هشدارها خاموش می‌شوند و `RANDOM_STATE = 42` همراه با `np.random.seed` برای تکرارپذیری نتایج تنظیم می‌شود.

</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully")

<div dir="rtl" style="text-align: right">

### خواندن دیتاست فعال
تابع `load_dataset` دیتاست انتخاب‌شده را می‌خواند. فرمت خام هر دیتاست فرق دارد (Colon به‌صورت ARFF، WDBC بدون هدر، Parkinson با دو ردیف هدر، TCGA از دو فایل داده و برچسب) و این تابع همه را به یک جدول با ستون هدف `class` تبدیل می‌کند (برای Parkinson ستون `group` یعنی شناسه‌ی فرد هم هست). اگر فایل `<نام دیتاست>_dataset.csv` موجود باشد، همان خوانده می‌شود. ابعاد و چند ردیف اول چاپ می‌شود.

</div>

In [ ]:
# Load the active dataset (path/format handled by dataset_config.py)
df = load_dataset(PROJECT_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

<div dir="rtl" style="text-align: right">

### فهرست فایل‌های پوشه‌ی datasets
همه‌ی پوشه‌ها و فایل‌های داخل `datasets` چاپ می‌شود تا مطمئن شویم فایل‌های هر دیتاست در جای درست قرار دارند. (این سلول برای پیدا کردن خطای مسیر فایل اضافه شده بود.)

</div>

In [ ]:
# اینجا یه ارور بود که مسیر رو فایل تنظیم نشده بود رو فولدر بوده
import os

DATASETS_PATH = PROJECT_PATH / "datasets"

for root, dirs, files in os.walk(DATASETS_PATH):
    print(f"\nFolder: {root}")

    for file in files:
        print("File:", file)

<div dir="rtl" style="text-align: right">

### خلاصه‌ی ابعاد داده
تعداد نمونه‌ها و تعداد ویژگی‌ها چاپ می‌شود. تعداد ویژگی‌ها برابر تعداد ستون‌ها منهای ستون هدف (و منهای ستون `group` در صورت وجود) است.

</div>

In [ ]:
print("Number of samples:", df.shape[0])
print("Number of features:", df.shape[1] - 1 - ("group" in df.columns))
print("Target column: class")
print("Dataset shape:", df.shape)

<div dir="rtl" style="text-align: right">

### جدا کردن ویژگی‌ها و برچسب
`split_xy` داده را به سه بخش تقسیم می‌کند: `X` (فقط ویژگی‌ها)، `y` (برچسب کلاس) و `groups` (شناسه‌ی افراد، فقط برای Parkinson). برچسب‌ها برای شمارش و رسم نمودار به رشته تبدیل می‌شوند. ابعاد و پنج برچسب اول نمایش داده می‌شود.

</div>

In [ ]:
X, y, groups = split_xy(df)
y = y.astype(str)

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nFirst 5 labels:")
print(y.head())

<div dir="rtl" style="text-align: right">

### توزیع کلاس‌ها
تعداد و درصد نمونه‌های هر کلاس محاسبه می‌شود. این کار نشان می‌دهد داده متوازن است یا نامتوازن؛ اگر نامتوازن باشد، باید از معیارهایی مثل F1 ماکرو و تقسیم طبقه‌ای (stratified) استفاده کرد.

</div>

In [ ]:
class_counts = y.value_counts()

print(class_counts)

print("\nClass percentages:")
print(y.value_counts(normalize=True) * 100)

<div dir="rtl" style="text-align: right">

### نمودار توزیع کلاس‌ها
همان تعداد کلاس‌ها در قالب نمودار میله‌ای نمایش داده می‌شود تا نامتوازن بودن داده یک‌نگاه دیده شود.

</div>

In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(
    x=y,
    hue=y,
    palette="Set2",
    legend=False
)

plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Samples")

plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right">

### بررسی داده‌ی گم‌شده
تعداد کل مقادیر خالی و ستون‌هایی که مقدار گم‌شده دارند چاپ می‌شود. اگر وجود داشته باشد باید قبل از مدل‌سازی پر یا حذف شوند.

</div>

In [ ]:
missing_values = df.isnull().sum()

print("Total missing values:", missing_values.sum())

print("\nColumns with missing values:")
print(missing_values[missing_values > 0])

<div dir="rtl" style="text-align: right">

### بررسی نوع داده‌ها
نوع ستون‌ها (عددی یا غیرعددی) و نوع ستون هدف چک می‌شود. انتظار داریم همه‌ی ویژگی‌ها عددی باشند، چون مدل‌ها به ورودی عددی نیاز دارند.

</div>

In [ ]:
print(df.dtypes.value_counts())

print("\nTarget data type:")
print(df["class"].dtype)

<div dir="rtl" style="text-align: right">

### ویژگی‌های ثابت
ستون‌هایی که فقط یک مقدار یکتا دارند پیدا می‌شوند. چنین ویژگی‌هایی هیچ اطلاعاتی برای جداسازی کلاس‌ها ندارند و بعداً در پیش‌پردازش (`drop_near_constant`) از روی داده‌ی train حذف می‌شوند.

</div>

In [ ]:
constant_features = X.columns[
    X.nunique() <= 1
]

print("Number of constant features:", len(constant_features))

print("Constant feature names:")
print(list(constant_features))

<div dir="rtl" style="text-align: right">

### ویژگی‌های تکراری
ستون‌هایی که مقادیرشان دقیقاً با ستون دیگری یکی است پیدا می‌شوند (با ترانهاده کردن جدول و `duplicated`). از هر گروه تکراری، نسخه‌ی اول نگه داشته می‌شود و بقیه شمرده می‌شوند. در پیش‌پردازش این کار فقط روی داده‌ی train انجام می‌شود.

</div>

In [ ]:
duplicate_features = X.columns[
    X.T.duplicated()
]

print("Number of duplicate features:", len(duplicate_features))

print("Duplicate feature names:")
print(list(duplicate_features))

<div dir="rtl" style="text-align: right">

### آمار توصیفی ویژگی‌ها
برای هر ویژگی تعداد، میانگین، انحراف معیار، کمینه، چارک‌ها و بیشینه نمایش داده می‌شود تا ایده‌ای از بازه و پراکندگی هر ویژگی داشته باشیم.

</div>

In [ ]:
display(X.describe().T)

<div dir="rtl" style="text-align: right">

### خلاصه‌ی مقیاس کل داده
میانگینِ میانگین‌ها و انحراف معیارها و کمترین و بیشترین مقدار کل داده چاپ می‌شود. اگر مقیاس ویژگی‌ها خیلی متفاوت باشد، نیاز به استانداردسازی (`StandardScaler`) داریم که در مدل‌ها استفاده می‌شود.

</div>

In [ ]:
print("Mean of feature means:", X.mean().mean())

print("Mean of feature standard deviations:", X.std().mean())

print("Minimum value:", X.min().min())

print("Maximum value:", X.max().max())

<div dir="rtl" style="text-align: right">

### هیستوگرام شش ویژگی نمونه
شش ویژگی با فاصله‌ی مساوی در میان ستون‌ها انتخاب و توزیع مقادیرشان (هیستوگرام همراه با منحنی KDE) رسم می‌شود. این نمودار چولگی، دامنه و داده‌های پرت را نشان می‌دهد.

</div>

In [ ]:
selected_features = [X.columns[i] for i in np.linspace(0, X.shape[1] - 1, 6).astype(int)]

fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 8)
)

for ax, feature in zip(axes.flatten(), selected_features):

    sns.histplot(
        X[feature],
        kde=True,
        ax=ax
    )

    ax.set_title(f"Distribution of {feature}")

plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right">

### ذخیره‌ی داده به‌صورت CSV
جدول کامل داده در پوشه‌ی `datasets` با نام `<نام دیتاست>_dataset.csv` ذخیره می‌شود تا نوت‌بوک‌های بعدی بدون پردازش دوباره‌ی فایل خام، سریع‌تر آن را بخوانند.

</div>

In [ ]:
OUTPUT_PATH = PROJECT_PATH / "datasets" / f"{ACTIVE_DATASET}_dataset.csv"

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("CSV file saved successfully!")
print("Saved at:", OUTPUT_PATH)

<div dir="rtl" style="text-align: right">

### ذخیره‌ی دوباره و تعریف `CSV_PATH`
مثل سلول قبل داده در همان مسیر ذخیره می‌شود (تکراری است و ضرری ندارد) و متغیر `CSV_PATH` را برای سلول بعد می‌سازد.

</div>

In [ ]:
CSV_PATH = PROJECT_PATH / "datasets" / f"{ACTIVE_DATASET}_dataset.csv"

df.to_csv(CSV_PATH, index=False)

print("Dataset saved successfully!")
print("Saved path:", CSV_PATH)
print("Dataset shape:", df.shape)

<div dir="rtl" style="text-align: right">

### بررسی فایل ذخیره‌شده
فایل ذخیره‌شده دوباره خوانده می‌شود و ابعادش با داده‌ی اصلی مقایسه می‌شود تا مطمئن شویم ذخیره درست انجام شده است.

</div>

In [ ]:
import pandas as pd

df_check = pd.read_csv(CSV_PATH)

print("Loaded shape:", df_check.shape)
print("Original shape:", df.shape)
print("File exists:", CSV_PATH.exists())